### 1. 패키지 설치 + 환경변수 로드

In [ ]:
%pip install -qU langchain langchain_openai langgraph

In [ ]:
from dotenv import load_dotenv
load_dotenv()

### 2. 그래프 준비

되감기 지점을 눈으로 확인하기 쉽도록 두 단계로 나눈 그래프를 씀

In [ ]:
from typing_extensions import TypedDict, NotRequired
from langchain_openai import ChatOpenAI
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.graph import StateGraph, START, END

class State(TypedDict):
    topic: str
    joke: NotRequired[str]   # 초기 입력에는 없고 write_joke 실행 후 채워짐

llm = ChatOpenAI(model="gpt-4o-mini")

def pick_topic(state: State):
    return {"topic": state["topic"]}

def write_joke(state: State):
    answer = llm.invoke(f"{state['topic']}에 대한 짧은 농담 하나만 만들어줘")
    return {"joke": answer.content}

graph_builder = StateGraph(State)
graph_builder.add_node("pick_topic", pick_topic)
graph_builder.add_node("write_joke", write_joke)
graph_builder.add_edge(START, "pick_topic")
graph_builder.add_edge("pick_topic", "write_joke")
graph_builder.add_edge("write_joke", END)

memory = InMemorySaver()
graph = graph_builder.compile(checkpointer=memory)

### 3. 최초 실행

In [ ]:
from langchain_core.runnables import RunnableConfig

config: RunnableConfig = {"configurable": {"thread_id": "1"}}

result = graph.invoke({"topic": "고양이"}, config)
print(result["joke"])

### 4. 되감을 지점 찾기

`get_state_history()` 에서 **아직 write_joke를 실행하기 직전** 체크포인트를 고름

In [ ]:
history = list(graph.get_state_history(config))

for state in history:
    meta = state.metadata or {}
    print(f"step={meta.get('step', ''):>2} next={str(state.next):<16} topic={state.values.get('topic')}")

before_joke = next(state for state in history if state.next == ("write_joke",))
print("\n선택한 체크포인트:", before_joke.config.get("configurable", {}).get("checkpoint_id"))

### 5. Replay — 같은 지점에서 다시 실행

해당 체크포인트의 config로 `invoke(None, ...)` 하면 그 시점부터 재실행됨

> Replay는 캐시를 읽는 것이 아니라 **노드를 다시 실행**함. LLM 호출도 다시 일어나므로 결과가 달라질 수 있음

In [ ]:
replay_result = graph.invoke(None, before_joke.config)   # 입력은 None, config가 시점을 지정
print(replay_result["joke"])

### 6. Fork — 상태를 바꿔서 다른 분기 만들기

`update_state()` 는 되돌리는 것이 아니라 **그 지점에서 갈라지는 새 체크포인트를 생성**함

In [ ]:
fork_config = graph.update_state(
    before_joke.config,
    values={"topic": "강아지"},   # 주제를 바꿔서 분기
)

fork_result = graph.invoke(None, fork_config)
print(fork_result["topic"], "→", fork_result["joke"])

### 7. 원본 히스토리 확인

fork를 해도 기존 체크포인트는 지워지지 않고 그대로 남아 있음

In [ ]:
for state in graph.get_state_history(config):
    meta = state.metadata or {}
    print(f"step={meta.get('step', ''):>2} source={meta.get('source', ''):<6} topic={state.values.get('topic')}")

### 8. 정리

- **Replay**: 과거 checkpoint_id의 config로 `invoke(None, config)` → 그 시점부터 재실행
- **Fork**: `update_state(과거 config, values=...)` 로 새 분기 생성 후 `invoke(None, fork_config)`
- `update_state` 는 **롤백이 아니라 분기**임. 원본 히스토리는 보존됨
- 참고: [Use time-travel](https://docs.langchain.com/oss/python/langgraph/use-time-travel)